<a href="https://colab.research.google.com/github/saugatme/SepsisPrediction/blob/main/assignment2_saugat_bhattarai_sepsispred_custom.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score
)
import warnings
warnings.filterwarnings('ignore')

In [2]:
torch.manual_seed(42)
np.random.seed(42)

In [3]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [4]:
from google.colab import drive
drive.mount('/content/drive')

In [5]:
#  Original 43 clinical features
FEATURE_COLUMNS = [
    'respiratory_minute_volume', 'heart_rate', 'leukocytes', 'temperature',
    'partial_co2', 'respiratory_rate', 'arterial_ph', 'bilirubin',
    'blood_urea_nitrogen', 'creatinine', 'diastolic_bp', 'fraction_of_inspired_o2',
    'mean_bp', 'partial_pressure_art._o2', 'systolic_bp', 'thrombocytes',
    'horowitz_index', 'bun/creatinine_ratio', 'delta-temperature',
    'lactate', 'bicarbonate', 'c-reactive_protein', 'hemoglobin',
    'heart_time_volume', 'lymphocytes', 'sodium', 'pancreatic_lipase',
    'procalcitonin', 'quick_score', 'oxygen_saturation', 'blood_glucose',
    'base_excess', 'chloride', 'calcium', 'potassium',
    'mixed_venous_oxygen_saturation', 'urine_output', 'net balance',
    'alanine_transaminase', 'aspartate_transaminase', 'stroke_volume',
    'svri', 'age'
]

Fastest-changing and most clinically informative markers before sepsis onset:
1. Lactate rises as cells starve of oxygen.
2. MAP drops as the cardiovascular system fails.
3. pH falls as organs release acid.
4. Procalcitonin spikes specifically with bacterial infection.

In [6]:
TREND_MARKERS = ['lactate', 'mean_bp', 'arterial_ph', 'procalcitonin']

#  4 markers with 2 time windows
TREND_FEATURES = [f'{col}_delta_{h}hr'for col in TREND_MARKERS for h in [1, 2]]
ALL_FEATURES = FEATURE_COLUMNS + TREND_FEATURES

### Preprocessing

In [7]:
def add_trend_features(df):
    df = df.copy()
    for col in TREND_MARKERS:
        df[f'{col}_delta_1hr'] = (df.groupby('id')[col].transform(lambda x: x.diff(2)))
        df[f'{col}_delta_2hr'] = (df.groupby('id')[col].transform(lambda x: x.diff(4)))
    return df

Three steps:
1. Forward-fill within each patient : Last known value is the best estimate until a new measurement is taken.
2. Median impute remaining gaps: Some patients never had certain labs measured at all.
3. Add trend features: Gives the model temporal awareness without a sequence model.

In [8]:
def preprocess_data(df):

    df = df.copy()

    # Forward fill per patient
    if 'id' in df.columns:
        df[FEATURE_COLUMNS] = df.groupby('id')[FEATURE_COLUMNS].ffill()

    # Median impute
    imputer = SimpleImputer(strategy='median')
    df[FEATURE_COLUMNS] = imputer.fit_transform(df[FEATURE_COLUMNS])

    # Trend features
    df = add_trend_features(df)
    for col in TREND_FEATURES:
        df[col] = df[col].fillna(0)

    return df

In [10]:
partitions = {}
for name in ['A', 'B', 'C', 'D']:
    partitions[name] = pd.read_csv(
        # f"D:/Data-Driven Health/Sem 2A/Assignment2_Sepsis/raw_data/sepsisexp_timeseries_partition-{name}.tsv", sep='\t'
        # Colab:
        f'/content/drive/MyDrive/ML Dataset/raw_data/sepsisexp_timeseries_partition-{name}.tsv', sep='\t')
    partitions[name] = preprocess_data(partitions[name])

For each timestep `i` in a patient's timeline:

#### Input Features
- `X_i`: 51 features at timestep `i`  
  - Includes the current patient snapshot and temporal trend features

#### Targets

- `y_severity` (Primary Target)
  - Continuous or ordinal severity level at the forecast horizon (N hours ahead).  
  - Represents the predicted position on the clinical severity scale after `N` hours.

- `y_binary` (Evaluation Target)
  - Binary indicator derived from `y_severity`:
  
  ```text
  y_binary = 1  if severity > 0 at horizon
  y_binary = 0  otherwise

In [11]:
def create_labels(df, horizon_hours):

    horizon_steps = int(horizon_hours * 2)  # 2hr → 4 steps, 4hr → 8 steps, 6hr → 12 steps
    X, y_severity, y_binary = [], [], []

    for patient_id in df['id'].unique():
        pat = (df[df['id'] == patient_id].sort_values('timestep').reset_index(drop=True))
        n = len(pat)

        for i in range(n - horizon_steps):
            # Input: current features
            features = pat.iloc[i][ALL_FEATURES].values

            # Target: severity at forecast point
            sev = pat.iloc[i + horizon_steps]['severity']

            X.append(features)
            y_severity.append(sev)
            y_binary.append(int(sev > 0))

    return (
        np.array(X,          dtype=np.float32),
        np.array(y_severity, dtype=np.float32),
        np.array(y_binary,   dtype=np.float32),)

We employ a 3-layer MLP for sepsis severity prediction.

Reason for MLP:
1. The temporal signal is already captured in our trend features
2. An MLP on 51 features is simpler, faster to train, and easier to explain with comparable accuracy for tabular clinical data.

Dual Output:
1. Severity head : predicts severity class (0-4) at forecast horizon
2. Binary head   : predicts sepsis yes/no (severity > 0)

Xavier uniform initialisation:  Xavier sets initial weights based on the size of each layer so gradients stay in a healthy range throughout training.

In [ ]:
class SepsisNet(nn.Module):
    def __init__(self, input_size=51):
        super().__init__()

        self.fc1   = nn.Linear(input_size, 64)         # Layer 1: 51 → 64
        self.bn1   = nn.BatchNorm1d(64)
        self.drop1 = nn.Dropout(0.3)

        self.fc2   = nn.Linear(64, 32)        # Layer 2: 64 → 32
        self.bn2   = nn.BatchNorm1d(32)
        self.drop2 = nn.Dropout(0.3)

        # Severity head: 32 → 5
        self.severity_head = nn.Linear(32, 5)         # 5 outputs = one score per severity class (0, 1, 2, 3, 4)
        self.binary_head = nn.Linear(32, 1)         # Binary head: 32 → 1 # Single probability: will severity > 0
        self._init_weights()         # Initialise all weights


    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        # Pass through shared backbone
        x = self.drop1(F.relu(self.bn1(self.fc1(x))))  
        x = self.drop2(F.relu(self.bn2(self.fc2(x))))  

        # Two separate outputs from the same representation
        severity_logits = self.severity_head(x)
        binary_prob     = torch.sigmoid(self.binary_head(x))

        return severity_logits, binary_prob

Compute class weights to handle imbalance: Class weights penalise the model more for getting rare classes wrong.

Two losses added together:

1. `CrossEntropyLoss` on severity (0–4): Standard loss for multi-class classification. It takes raw logits, applies softmax internally, measures how wrong the predicted class distribution is.
2. `BCELoss` on binary (0/1): Standard for binary classification. It measures how far the sigmoid probability is from the true label.

Combined beacause: The binary loss acts as extra supervision i.e. it pushes the model to clearly separate sick vs healthy, which helps the severity task too.

In [13]:
def get_class_weights(y_severity, n_classes=5):
    counts = np.bincount(y_severity.astype(int), minlength=n_classes)
    total  = len(y_severity)
    weights = total / (n_classes * counts.clip(min=1))
    return torch.FloatTensor(weights).to(DEVICE)


def combined_loss(sev_logits, bin_prob, y_sev, y_bin, class_weights):
    loss_severity = F.cross_entropy(sev_logits, y_sev.long(), weight=class_weights)
    loss_binary = F.binary_cross_entropy(bin_prob.squeeze(), y_bin)
    return loss_severity + 0.3 * loss_binary

- `StandardScaler` ensures all 51 features are on equal footing.
-  batch_size=256:
    - Larger batches = more stable gradient estimates.
    - 256 is a good balance between fast training and stable updates.
    - `shuffle=True` ensures the model doesn't learn the order of patients.

-  `Adam Optimizer`: Adapts learning rate per parameter automatically
    - More reliable than plain SGD for tabular medical data.
    - `weight_decay=1e-4` is L2 regularisation ; Small penalty on large weights -> reduces overfitting.

# Validation on AUROC not loss because:
- Loss can keep improving even when the model isn't getting better at the actual task.
- AUROC directly measures ranking quality which is what we care about clinically.

In [14]:
def train_model(train_data, val_data, horizon_hours):

    #  Build arrays
    X_tr, y_sev_tr, y_bin_tr = create_labels(train_data, horizon_hours)
    X_va, y_sev_va, y_bin_va = create_labels(val_data, horizon_hours)

    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr)
    X_va = scaler.transform(X_va)
    class_weights = get_class_weights(y_sev_tr)     #  Class weights from training labels only

    #  DataLoader
    train_ds = TensorDataset(
        torch.FloatTensor(X_tr),
        torch.FloatTensor(y_sev_tr),
        torch.FloatTensor(y_bin_tr)
    )
    train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)

    model = SepsisNet(input_size=X_tr.shape[1]).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

    # Training loop
    best_auroc      = 0.0
    best_state      = None
    patience_counter = 0
    PATIENCE        = 15

    for epoch in range(100):
        model.train()
        for batch_X, batch_sev, batch_bin in train_loader:
            batch_X   = batch_X.to(DEVICE)
            batch_sev = batch_sev.to(DEVICE)
            batch_bin = batch_bin.to(DEVICE)

            sev_logits, bin_prob = model(batch_X)
            loss = combined_loss(sev_logits, bin_prob, batch_sev, batch_bin, class_weights)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # Validate
        model.eval()
        with torch.no_grad():
            X_va_t = torch.FloatTensor(X_va).to(DEVICE)
            _, bin_prob_va = model(X_va_t)
            probs_va = bin_prob_va.cpu().numpy().squeeze()

        try:
            val_auroc = roc_auc_score(y_bin_va, probs_va)
        except Exception:
            val_auroc = 0.5

        if epoch % 10 == 0:
            print(f'  Epoch {epoch:3d} | Val AUROC: {val_auroc:.4f}')

        #  Early stopping
        if val_auroc > best_auroc:
            best_auroc   = val_auroc
            best_state   = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f'  Early stopping at epoch {epoch} '
                      f'(best AUROC: {best_auroc:.4f})')
                break

    # Restore best weights before returning
    model.load_state_dict(best_state)
    return model, scaler

- Each fold trains on 3 partitions, tests on 1.
- Rotating all 4 gives us results across the full dataset, not just one train/test split.
    - Averaging across folds gives a reliable estimate of true model performance.

In [15]:
def cross_validation(partitions, horizon_hours):
    folds = [
        {'train': ['A','B','C'], 'test': 'D'},
        {'train': ['A','B','D'], 'test': 'C'},
        {'train': ['A','C','D'], 'test': 'B'},
        {'train': ['B','C','D'], 'test': 'A'},
    ]

    all_results = []
    saved_models = []

    for i, fold in enumerate(folds):
        print(f'\nFold {i+1}/4 — test: partition {fold["test"]}')
        print('-' * 45)

        train_data = pd.concat(
            [partitions[p] for p in fold['train']],
            ignore_index=True
        )
        test_data = partitions[fold['test']]

        model, scaler = train_model(train_data, test_data, horizon_hours)
        results       = evaluate_model(model, scaler, test_data, horizon_hours)

        all_results.append(results)
        saved_models.append((model, scaler))

        torch.save(
            model.state_dict(),
            f'model_{horizon_hours}hr_fold{i+1}.pth'
        )

    metrics = ['auroc', 'auprc', 'precision', 'recall', 'f1', 'severity_mae']
    avg = {m: np.mean([r[m] for r in all_results]) for m in metrics}
    std = {m: np.std( [r[m] for r in all_results]) for m in metrics}

    print(f'\n{"="*45}')
    print(f'CV Results — {horizon_hours}hr horizon')
    for m in metrics:
        print(f'  {m:12s}: {avg[m]:.3f} ± {std[m]:.3f}')
    print(f'{"="*45}')

    return avg, std, saved_models

Two separate evaluations:
1. Binary metrics (AUROC, AUPRC, Recall, F1) from binary head
2. Severity MAE from severity head

- MAE because Severity is ordinal (0,1,2,3,4).
    - Being off by 1 level is much better than being off by 3. MAE captures this; it rewards being close, not just exact.

In [16]:
def evaluate_model(model, scaler, test_data, horizon_hours, threshold=0.5):
    X_test, y_sev, y_bin = create_labels(test_data, horizon_hours)
    X_test = scaler.transform(X_test)

    model.eval()
    with torch.no_grad():
        X_t = torch.FloatTensor(X_test).to(DEVICE)
        sev_logits, bin_prob = model(X_t)

        # Binary predictions
        probs = bin_prob.cpu().numpy().squeeze()
        preds = (probs > threshold).astype(int)

        # Severity predictions
        sev_preds = sev_logits.argmax(dim=1).cpu().numpy()

    results = {
        #  Binary metrics
        'auroc':     roc_auc_score(y_bin, probs),
        'auprc':     average_precision_score(y_bin, probs),
        'precision': precision_score(y_bin, preds, zero_division=0),
        'recall':    recall_score(y_bin, preds, zero_division=0),
        'f1':        f1_score(y_bin, preds, zero_division=0),

        #  Severity metric
        'severity_mae': np.mean(np.abs(sev_preds - y_sev)),
    }

    print(f'  AUROC={results["auroc"]:.3f} | '
          f'AUPRC={results["auprc"]:.3f} | '
          f'Recall={results["recall"]:.3f} | '
          f'F1={results["f1"]:.3f} | '
          f'SevMAE={results["severity_mae"]:.3f}')

    return results

- For each feature: shuffle it, measure AUROC drop.
- Importance = how much AUROC drops when that feature is destroyed.

Smallest set of features that retains 95% of baseline AUROC.

- Why 95%: A small accuracy trade-off for a much simpler model is worth it clinically
    - fewer features = faster measurement, lower cost, easier to deploy at bedside.

- Method: Add features one by one in importance order, stop when we hit the 95% retention target.


In [17]:
def permutation_importance(model, scaler, test_data, horizon_hours):
    X_test, _, y_bin = create_labels(test_data, horizon_hours)
    X_test_sc = scaler.transform(X_test)

    # Baseline — normal predictions
    model.eval()
    with torch.no_grad():
        _, bin_prob = model(torch.FloatTensor(X_test_sc).to(DEVICE))
        base_probs  = bin_prob.cpu().numpy().squeeze()
    baseline_auroc = roc_auc_score(y_bin, base_probs)

    importances = []

    for fi, feat_name in enumerate(ALL_FEATURES):
        # Shuffle just this one feature column
        X_shuffled = X_test_sc.copy()
        np.random.shuffle(X_shuffled[:, fi])

        with torch.no_grad():
            _, bin_prob = model(torch.FloatTensor(X_shuffled).to(DEVICE))
            shuffled_probs = bin_prob.cpu().numpy().squeeze()

        shuffled_auroc = roc_auc_score(y_bin, shuffled_probs)
        drop = baseline_auroc - shuffled_auroc  # positive = feature was useful

        importances.append({
            'feature':    feat_name,
            'importance': drop
        })

    # Sort
    importances.sort(key=lambda x: x['importance'], reverse=True)

    print(f'\nTop 15 features — {horizon_hours}hr horizon '
          f'(baseline AUROC={baseline_auroc:.4f})')
    print('-' * 28)
    for i, item in enumerate(importances[:15], 1):
        bar = '█' * max(0, int(item['importance'] * 300))
        print(f'{i:2d}. {item["feature"]:35s} {item["importance"]:+.4f}  {bar}')

    return importances


def find_minimal_features(model, scaler, test_data, horizon_hours, importances, threshold=0.95):
    X_test, _, y_bin = create_labels(test_data, horizon_hours)
    X_test_sc = scaler.transform(X_test)

    model.eval()
    with torch.no_grad():
        _, bin_prob = model(torch.FloatTensor(X_test_sc).to(DEVICE))
        base_probs  = bin_prob.cpu().numpy().squeeze()
    baseline_auroc = roc_auc_score(y_bin, base_probs)
    target_auroc   = baseline_auroc * threshold

    feat_index = {f: i for i, f in enumerate(ALL_FEATURES)}
    ranked     = [item['feature'] for item in importances]

    for k in range(1, len(ranked) + 1):
        # Zero out all features except top-k
        X_reduced = np.zeros_like(X_test_sc)
        for feat in ranked[:k]:
            fi = feat_index[feat]
            X_reduced[:, fi] = X_test_sc[:, fi]

        with torch.no_grad():
            _, bin_prob = model(torch.FloatTensor(X_reduced).to(DEVICE))
            probs_k     = bin_prob.cpu().numpy().squeeze()
        auroc_k = roc_auc_score(y_bin, probs_k)

        if auroc_k >= target_auroc:
            print(f'\n{horizon_hours}hr minimal set: {k} features '
                  f'(AUROC={auroc_k:.4f}, target={target_auroc:.4f})')
            print(f'Features: {ranked[:k]}')
            return ranked[:k]

    return ranked

In [18]:
all_results = {}
all_models  = {}

for h in [2, 4, 6]:
    print(f'\n{"#"*50}')
    print(f'  {h}-hour prediction')
    print(f'{"#"*50}')

    avg, std, saved_models = cross_validation(partitions, horizon_hours=h)
    all_results[h] = (avg, std)
    all_models[h]  = saved_models

    # Fold 4 = index 3 = trained on B,C,D, tested on A
    model_f4, scaler_f4 = all_models[h][3]
    importances = permutation_importance(model_f4, scaler_f4, partitions['A'], h)
    minimal = find_minimal_features(model_f4, scaler_f4, partitions['A'], h, importances)

# Print final summary
print('\nFINAL SUMMARY')
print('=' * 55)
print(f'{"Horizon":<10} {"AUROC":<14} {"Recall":<14} {"SevMAE"}')
print('-' * 55)
for h in [2, 4, 6]:
    avg, std = all_results[h]
    print(f'{h}hr        '
          f'{avg["auroc"]:.3f}±{std["auroc"]:.3f}   '
          f'{avg["recall"]:.3f}±{std["recall"]:.3f}   '
          f'{avg["severity_mae"]:.3f}±{std["severity_mae"]:.3f}')